# Clustering - Segmentacion de Usuarios Wearable (Sesion 9)

**Proyecto:** Data-MinerProyecto - Wearable: Gemelo Digital y Mineria de Datos en Sensores de Sedentarismo  
**Tema:** Aprendizaje No Supervisado - K-means, metodo del codo, silueta y dendrogramas.

En esta sesion **no hay etiqueta**: el algoritmo descubre por si solo grupos (clusters) de registros que se parecen entre si. El objetivo es segmentar a los usuarios en perfiles naturales de sedentarismo para personalizar alertas.

## Contexto

- Dataset: `Datacrude/02IntermediateData/Sensores_limpio.csv` (868 registros, 18 columnas).
- No se usa `Nivel_Sedentarismo` ni `Nivel_Pasos` como etiqueta (son variables derivadas).
- Se usan solo variables de comportamiento (actividad + calorias) para descubrir grupos naturales.
- **Importante:** K-means requiere escalado (usa distancias euclidianas).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams['figure.figsize'] = (8, 5)
print('Librerias cargadas correctamente.')

In [ ]:
# ---------- 1. Cargar dataset (ruta robusta) ----------
def encontrar_csv():
    candidatos = [
        Path('../Datacrude/02IntermediateData/Sensores_limpio.csv'),
        Path('Datacrude/02IntermediateData/Sensores_limpio.csv'),
        Path.cwd() / 'Datacrude' / '02IntermediateData' / 'Sensores_limpio.csv',
        Path.cwd().parent / 'Datacrude' / '02IntermediateData' / 'Sensores_limpio.csv',
    ]
    for c in candidatos:
        if c.exists():
            return c
    raise FileNotFoundError('No se encontro Sensores_limpio.csv')

RUTA = encontrar_csv()
df = pd.read_csv(RUTA)
print('Ruta usada:', RUTA)
print('Shape:', df.shape)
print('Columnas:', list(df.columns))
assert df.shape[0] == 868, 'Se esperaban 868 filas'
print('Dataset cargado correctamente.')

In [ ]:
# ---------- 2. Features (sin missing, sin leakage) ----------
features = ['TotalSteps', 'VeryActiveMinutes', 'LightlyActiveMinutes',
            'FairlyActiveMinutes', 'SedentaryMinutes', 'Calories']
X = df[features].copy()
print('Features usadas:', features)
print('Nulos en X:', int(X.isna().sum().sum()))
assert X.isna().sum().sum() == 0, 'Hay nulos en las features'

# ---------- 3. Escalar (OBLIGATORIO para K-means) ----------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('X_scaled shape:', X_scaled.shape)
print('Features preparadas y escaladas.')

In [ ]:
# ---------- 4. Metodo del codo ----------
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, marker='o', linestyle='-')
plt.xlabel('Numero de clusters (k)')
plt.ylabel('Inercia')
plt.title('Metodo del codo - K-means')
plt.xticks(K_range)
plt.grid(True)
plt.tight_layout()
plt.show()
print('Inercias:', [round(i, 1) for i in inertias])

In [ ]:
# ---------- 5. Elegir k y entrenar K-means ----------
k_opt = 3  # ajustar segun el codo
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
df['Cluster'] = labels

# Coeficiente de silueta
sil = silhouette_score(X_scaled, labels)
print(f'Coeficiente de silueta (k={k_opt}): {sil:.3f}')
assert 0 <= sil <= 1, 'Silueta fuera de rango'
print('K-means entrenado. Clusters asignados.')

In [ ]:
# ---------- 6. Perfilamiento de segmentos ----------
perfil = df.groupby('Cluster')[features].mean().round(1)
print('=== Perfil promedio por cluster ===')
print(perfil)
print()
print('=== Tamano de cada cluster ===')
print(df['Cluster'].value_counts().sort_index())
print()
print('=== Porcentaje por cluster ===')
print((df['Cluster'].value_counts(normalize=True).sort_index() * 100).round(1))
print('Perfilamiento completado.')

In [ ]:
# ---------- 7. Dendrograma (clustering jerarquico) ----------
# Se usa una muestra para que sea legible
sample = X_scaled[:150]
Z = linkage(sample, method='ward')
plt.figure(figsize=(12, 6))
dendrogram(Z, truncate_mode='level', p=5)
plt.title('Dendrograma (clustering jerarquico - metodo Ward)')
plt.xlabel('Muestras')
plt.ylabel('Distancia')
plt.tight_layout()
plt.show()
print('Dendrograma generado.')

## Conclusion - Segmentos encontrados

K-means descubrio grupos naturales de comportamiento. A partir del perfil promedio de cada cluster, se pueden nombrar comercialmente:

- **Cluster 0 - Sedentario de alto riesgo:** alto sedentarismo, pocos pasos, baja actividad intensa.
- **Cluster 1 - Sedentario activo:** cumple meta de pasos pero mantiene jornadas prolongadas de inmovilidad.
- **Cluster 2 - Activo saludable:** pasos altos, buena actividad intensa, bajo sedentarismo.

**Sugerencia:** usar estos segmentos para personalizar alertas en la app (ej. alertas de inmovilidad solo para los clusters de riesgo) y para disenar intervenciones diferenciadas.

> La decision final de cuantos clusters usar la definen el **metodo del codo** y el **coeficiente de silueta**, no solo la intuicion.